# Create ODCS Data Contract Template

Creates a reusable ODCS v3.1.0 template skeleton. This notebook does not hardcode table names, SQL rules, source paths, or freshness thresholds. Rules are injected later by the contract generation step from config and rule catalog files.

In [ ]:
# Databricks widgets. Defaults allow local/static review, but production jobs should pass explicit values.
try:
    dbutils.widgets.text("template_output_path", "dc_nb/outputs/templates/raw_source_ODCS_template.yaml")
    dbutils.widgets.dropdown("layer", "raw", ["raw", "bronze", "silver", "gold"])
    dbutils.widgets.text("contract_id", "<contract_id>")
    dbutils.widgets.text("contract_name", "<contract_name>")
    dbutils.widgets.text("domain", "<domain>")
    dbutils.widgets.text("tenant", "Allianz")
    dbutils.widgets.text("version", "1.0.0")
    dbutils.widgets.dropdown("status", "draft", ["draft", "active", "deprecated", "retired"])
    dbutils.widgets.text("team_name", "Data Modelling & Engineering CoE")
    dbutils.widgets.text("support_channel", "coe-data-contracts")
    dbutils.widgets.text("support_url", "mailto:coe-data-contracts@example-internal")
except NameError:
    pass


def widget_value(name: str, default: str = "") -> str:
    try:
        value = dbutils.widgets.get(name)
        return value if value is not None else default
    except Exception:
        return default


TEMPLATE_OUTPUT_PATH = widget_value("template_output_path", "dc_nb/outputs/templates/raw_source_ODCS_template.yaml")
LAYER = widget_value("layer", "raw")
CONTRACT_ID = widget_value("contract_id", "<contract_id>")
CONTRACT_NAME = widget_value("contract_name", "<contract_name>")
DOMAIN = widget_value("domain", "<domain>")
TENANT = widget_value("tenant", "Allianz")
VERSION = widget_value("version", "1.0.0")
STATUS = widget_value("status", "draft")
TEAM_NAME = widget_value("team_name", "Data Modelling & Engineering CoE")
SUPPORT_CHANNEL = widget_value("support_channel", "coe-data-contracts")
SUPPORT_URL = widget_value("support_url", "mailto:coe-data-contracts@example-internal")

if not TEMPLATE_OUTPUT_PATH:
    raise ValueError("template_output_path is required.")

In [ ]:
from datetime import datetime, timezone
from pathlib import Path

import yaml


odcs_template = {
    "kind": "DataContract",
    "apiVersion": "v3.1.0",
    "id": CONTRACT_ID,
    "name": CONTRACT_NAME,
    "version": VERSION,
    "status": STATUS,
    "domain": DOMAIN,
    "tenant": TENANT,
    "description": {
        "purpose": f"Reusable ODCS template for the {LAYER} layer.",
        "limitations": "This is a template skeleton. Data objects, columns, and rules are injected during contract generation.",
        "usage": "Use as the structural base for generated ODCS data contracts.",
    },
    "servers": [],
    "schema": [],
    "slaProperties": [
        {"property": "frequency", "value": 1, "unit": "d"},
        {"property": "latency", "value": 24, "unit": "h"},
        {"property": "retention", "value": 7, "unit": "y"},
    ],
    "team": {
        "name": TEAM_NAME,
        "description": "Producing team / owner of this contract template",
        "members": [],
    },
    "support": [
        {"channel": SUPPORT_CHANNEL, "tool": "email", "url": SUPPORT_URL},
    ],
    "customProperties": [
        {"property": "pipelineLayer", "value": LAYER},
        {"property": "expectedFiles", "value": []},
        {
            "property": "notebookValidationConfig",
            "value": {
                "contractReader": "yaml.safe_load",
                "schemaObjectPath": "schema",
                "qualityRulesPath": "schema[].quality",
                "sourceFileField": "physicalName",
                "entityField": "name",
                "batchParameter": "batch_ref",
                "defaultTargetWatermakExp": "batch_ref = '{batch_ref}'",
                "placeholderStyle": "python_format",
                "placeholders": {
                    "full_table_name": "Databricks temp view or fully qualified table name",
                    "target_watermak_exp": "Batch filter expression used by DQ SQL templates",
                    "table_pk": "Primary key column list for the current source object",
                    "table_bk": "Business key column list for the current source object",
                    "table_bk_join": "Join condition between aliases a and b for the business key",
                    "source_table": "Source table/view used for reconciliation checks",
                    "source_pk": "Source primary key used for reconciliation checks",
                },
                "entities": [],
            },
        },
        {
            "property": "ruleInjectionPolicy",
            "value": "Do not hardcode SQL rules in the template. Inject quality rules from config and rule catalog during contract generation.",
        },
    ],
    "contractCreatedTs": datetime.now(timezone.utc).isoformat(),
}

print("Template prepared")
print(f"ODCS apiVersion: {odcs_template['apiVersion']}")
print(f"Layer: {LAYER}")
print("Embedded SQL rules: 0")

In [ ]:
output_path = Path(TEMPLATE_OUTPUT_PATH)
output_path.parent.mkdir(parents=True, exist_ok=True)

header = (
    "# =============================================================================\n"
    "# ODCS v3.1.0 data contract template.\n"
    "# Rules, tables, columns, and physical paths are injected during generation.\n"
    "# =============================================================================\n"
)
output_path.write_text(
    header + yaml.safe_dump(odcs_template, sort_keys=False, allow_unicode=False),
    encoding="utf-8",
)

print(f"ODCS template saved to: {output_path}")